In [2]:
import os
import yaml
import pandas as pd
import re

# === CONFIG ===
PROJECTS_DIR = r"C:\Users\Admin\OneDrive\Education\Master of Info - Thesis\Config Files"
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUMMARY_CSV = os.path.join(OUTPUT_DIR, "project_emulator.csv")
DETAILED_CSV = os.path.join(OUTPUT_DIR, "project_emulators_detailed.csv")

# === TEST CLASSIFICATION KEYWORDS ===
TEST_TYPES = {
    'firebase_test_lab': ['firebase test', 'gcloud firebase test android run'],
    'appcenter_test': ['appcenter test run', 'microsoft/appcenter-test-cli-action'],
    'browserstack_test': ['browserstack', 'browserstack/github-actions'],
    'GitHub_emulator_full': ['android-emulator-runner'],
    'GitHub_emulator_compact': ['malinskiy/action-android/emulator-run-cmd'],
    'GitHub_emulator_manual': ['create avd'],
    'GitHub_gradle': ['connectedReleaseAndroidTest', 'connectedcheck', 'connectedDebugAndroidTest', 'connectedAndroidTest']
}

# === DETECT TEST TYPES ===
def detect_testing_types(yaml_text):
    # Remove commented lines
    uncommented_text = '\n'.join(
        line for line in yaml_text.splitlines()
        if not line.strip().startswith('#')
    ).lower()

    found = set()
    for label, keywords in TEST_TYPES.items():
        for kw in keywords:
            if kw.lower() in uncommented_text:
                found.add(label)
    return found

# === RECURSIVE SEARCH FOR 'emulator-options' VALUES ===
def extract_emulator_options(obj):
    options = []

    def recurse(o):
        if isinstance(o, dict):
            for k, v in o.items():
                if isinstance(k, str) and 'emulator-options' in k.lower():
                    if isinstance(v, str):
                        options.append(v)
                recurse(v)
        elif isinstance(o, list):
            for item in o:
                recurse(item)

    recurse(obj)
    return options

# === PARSE FLAGS: "-flag value" grouped as a unit ===
def parse_emulator_flags(option_str):
    tokens = re.findall(r'(-\S+(?:\s[^-]\S*)*)', option_str.strip())
    return [token.strip() for token in tokens]

# === PARSE YAML FILE ===
def parse_yaml_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw = f.read().replace('\t', ' ')
            test_types = detect_testing_types(raw)
            content = yaml.safe_load(raw)
            if not content:
                return {'types': test_types, 'options': [], 'error': True}
            emulator_options = extract_emulator_options(content)
            return {'types': test_types, 'options': emulator_options, 'error': False}
    except Exception:
        return {'types': set(), 'options': [], 'error': True}

# === SCAN PROJECTS ===
project_results = {}
detailed_rows = []

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)
            # Correct project name: between first and second dot
            project_name = filename.split('.')[1] if filename.count('.') >= 2 else filename.split('.')[0]

            result = parse_yaml_file(file_path)

            if project_name not in project_results:
                project_results[project_name] = {
                    'types': set(),
                    'parsed_flags': [],
                    'errors': 0,
                    'yml_count': 0
                }

            project_results[project_name]['types'].update(result['types'])
            project_results[project_name]['yml_count'] += 1
            if result['error']:
                project_results[project_name]['errors'] += 1

            # Parse all emulator-options for this file
            for opt_str in result['options']:
                parsed = parse_emulator_flags(opt_str)
                project_results[project_name]['parsed_flags'].extend(parsed)

# === BUILD DETAILED ROWS ===
for project, data in project_results.items():
    for flag in data['parsed_flags']:
        detailed_rows.append({
            'project': project,
            'emulator_option': f'"{flag}"',
            'yml_count': data['yml_count']
        })

# === EXPORT SUMMARY CSV ===
summary_rows = []
for project, data in project_results.items():
    summary_rows.append({
        'project': project,
        'test_types': ', '.join(sorted(data['types'])) if data['types'] else 'none',
        'emulator_option_items_count': len(data['parsed_flags']),
        'yml_count': data['yml_count'],
        'yaml_errors': data['errors']
    })

pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(detailed_rows).to_csv(DETAILED_CSV, index=False)

print(f"\n✅ Emulator summary CSV saved to: {SUMMARY_CSV}")
print(f"✅ Emulator detailed CSV saved to: {DETAILED_CSV}")



✅ Emulator summary CSV saved to: C:\GitHub\Android-Mobile-Apps\project_emulator.csv
✅ Emulator detailed CSV saved to: C:\GitHub\Android-Mobile-Apps\project_emulators_detailed.csv
